# Netflix Content Strategy & Audience Classification
### Applied Predictive Analytics — Case Study

**Dataset:** Netflix Movies and TV Shows (Kaggle — `shivamb/netflix-shows`)

**Goal:** Understand how Netflix's content catalog has grown and what it's
made of, then build a model that predicts a title's **target audience group**
(Kids/Family, Teens, or Adults) from its metadata — useful for catalog
curation, parental controls, and regional content strategy.

This notebook is organized into two parts, matching the presentation:

- **Part 1 — Understanding the problem & the data** (visualizations)
- **Part 2 — Algorithms used to build the audience-classification model**

Run the cells in order (Runtime → Run all) inside Google Colab.


## 0. Setup — Install & Import Libraries

In [ ]:
# Install kagglehub (only needed once per Colab session)
!pip install -q kagglehub


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", None)


## 1. Load the Dataset

We use **kagglehub** to pull the dataset directly from Kaggle inside Colab
(no manual download needed). The first time you run this in Colab, it may
ask you to authenticate with your Kaggle account — follow the on-screen
instructions (or upload your `kaggle.json` API token if prompted).

> If `kagglehub` download ever fails (e.g. no internet / auth issue), you can
> alternatively download `netflix_titles.csv` manually from
> https://www.kaggle.com/datasets/shivamb/netflix-shows , upload it to
> Colab (folder icon → upload), and change `csv_path` below to its uploaded filename.


In [ ]:
import kagglehub
import os

# Downloads the dataset and returns the local folder path
dataset_path = kagglehub.dataset_download("shivamb/netflix-shows")
print("Dataset downloaded to:", dataset_path)
print(os.listdir(dataset_path))


In [ ]:
csv_name = [f for f in os.listdir(dataset_path) if f.endswith(".csv")][0]
csv_path = os.path.join(dataset_path, csv_name)

df = pd.read_csv(csv_path)
print("Shape of dataset:", df.shape)
df.head()


## 2. Dataset Overview

The Netflix Titles dataset contains **~8,800 titles** on the platform, with:

- **`type`** — Movie or TV Show
- **`title`, `director`, `cast`** — descriptive metadata
- **`country`** — country/countries of production
- **`date_added`** — date the title was added to Netflix
- **`release_year`** — the title's original release year
- **`rating`** — official content rating (e.g. PG, TV-MA, TV-14)
- **`duration`** — runtime in minutes (Movies) or number of seasons (TV Shows)
- **`listed_in`** — genre(s) the title is categorized under

This is a **real-world, real-company dataset** describing exactly how a
major streaming platform's catalog is structured — genre mix, country
sourcing, and audience ratings are all decisions with real business impact.


In [ ]:
df.info()


In [ ]:
df.describe(include='all').T


## 3. Data Cleaning & Feature Engineering

We parse dates, split the mixed-format `duration` column (minutes for
Movies, seasons for TV Shows) into a usable number, extract the primary
country and genre count, and map the many specific `rating` values into
three broad audience groups.


In [ ]:
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
df['year_added'] = df['date_added'].dt.year

# Split the mixed-unit duration column into a numeric value
df['duration_value'] = df['duration'].str.extract(r'(\d+)').astype(float)

# Primary country = first country listed (many titles list several)
df['primary_country'] = df['country'].fillna('Unknown').apply(lambda x: x.split(',')[0].strip())

# Number of genres a title is tagged with
df['num_genres'] = df['listed_in'].fillna('').apply(lambda x: len([g for g in x.split(',') if g.strip()]))

# Map specific ratings into 3 broad audience groups
rating_map = {
    'G': 'Kids/Family', 'TV-Y': 'Kids/Family', 'TV-Y7': 'Kids/Family',
    'TV-Y7-FV': 'Kids/Family', 'TV-G': 'Kids/Family', 'PG': 'Kids/Family',
    'PG-13': 'Teens', 'TV-14': 'Teens',
    'R': 'Adults', 'TV-MA': 'Adults', 'NC-17': 'Adults',
}
df['audience_group'] = df['rating'].map(rating_map)

# Drop rows missing the fields we need for analysis/modeling
df_clean = df.dropna(subset=['audience_group', 'duration_value', 'year_added']).copy()

print("Original rows:", len(df))
print("Rows after cleaning (used for modeling):", len(df_clean))
df_clean[['type', 'release_year', 'rating', 'audience_group', 'duration_value', 'primary_country', 'num_genres']].head()


# Part 1 — Visualizations

Five visualizations are used to understand **what Netflix's catalog looks
like and how it has evolved**.


### Visualization 1 — Movies vs. TV Shows

What share of the catalog is movies versus TV shows?


In [ ]:
type_counts = df['type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

sns.countplot(x='type', data=df, palette=['#E50914', '#B3B3B3'], ax=axes[0])
axes[0].set_title('Movies vs. TV Shows — Counts')
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Titles')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2, p.get_height()),
                      ha='center', va='bottom', fontsize=10)

axes[1].pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
            colors=['#E50914', '#B3B3B3'], startangle=90, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Movies vs. TV Shows — Proportion')

plt.suptitle('Visualization 1: Content Type Split', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


**Insight:** Movies make up roughly **70%** of the catalog versus about
30% TV Shows — Netflix's library still leans heavily toward films, though
TV shows tend to drive longer-term subscriber retention through episodic
engagement.


### Visualization 2 — Catalog Growth Over Time

How has the number of titles added to Netflix changed year over year?


In [ ]:
yearly_adds = df.dropna(subset=['year_added']).groupby(['year_added', 'type']).size().unstack(fill_value=0)

plt.figure(figsize=(10, 5.5))
plt.stackplot(yearly_adds.index, yearly_adds['Movie'], yearly_adds['TV Show'],
              labels=['Movie', 'TV Show'], colors=['#E50914', '#564d4d'])
plt.title('Visualization 2: Titles Added to Netflix Per Year', fontsize=13, fontweight='bold')
plt.xlabel('Year Added')
plt.ylabel('Number of Titles Added')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()


**Insight:** Content additions grew explosively from **2015 to 2019** as
Netflix aggressively expanded its catalog, then leveled off — reflecting a
strategic shift from rapid library growth toward original, curated content.


### Visualization 3 — Top 10 Genres

Which genres dominate the Netflix catalog?


In [ ]:
genre_series = df['listed_in'].dropna().str.split(', ').explode()
top_genres = genre_series.value_counts().head(10)

plt.figure(figsize=(9, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, color='#E50914')
plt.title('Visualization 3: Top 10 Genres on Netflix', fontsize=13, fontweight='bold')
plt.xlabel('Number of Titles')
plt.ylabel('')
plt.tight_layout()
plt.show()


**Insight:** **International Movies, Dramas, and Comedies** dominate the
catalog — showing Netflix's dual strategy of broad mainstream appeal
alongside deliberate international/regional content investment.


### Visualization 4 — Content Rating (Audience Group) Distribution

How is the catalog split across Kids/Family, Teens, and Adult content?


In [ ]:
audience_counts = df_clean['audience_group'].value_counts().reindex(['Kids/Family', 'Teens', 'Adults'])

plt.figure(figsize=(8, 5.5))
sns.barplot(x=audience_counts.index, y=audience_counts.values, palette=['#F5B7B1', '#E85D5D', '#E50914'])
plt.title('Visualization 4: Content Distribution by Audience Group', fontsize=13, fontweight='bold')
plt.xlabel('')
plt.ylabel('Number of Titles')
for i, v in enumerate(audience_counts.values):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


**Insight:** **Adult-rated content (TV-MA / R / NC-17) is the largest
group**, followed by Teens, with Kids/Family content the smallest share —
suggesting the core catalog skews toward a mature, general-audience
strategy rather than a family-first one.


### Visualization 5 — Top 10 Content-Producing Countries

Which countries produce the most titles on Netflix?


In [ ]:
top_countries = df['primary_country'].value_counts().head(10)

plt.figure(figsize=(9, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, color='#E50914')
plt.title('Visualization 5: Top 10 Content-Producing Countries', fontsize=13, fontweight='bold')
plt.xlabel('Number of Titles')
plt.ylabel('')
plt.tight_layout()
plt.show()


**Insight:** The **United States dominates** as the top content source,
with India, the UK, and other countries following — reflecting Netflix's
global footprint while still being anchored in US-originated content.

---

## Part 1 Summary

The five visualizations above show that Netflix's catalog is shaped by:
1. **A movie-heavy library** with TV shows as a growing minority
2. **Explosive growth (2015-2019)** followed by a more measured pace
3. **Genre concentration** in international, drama, and comedy content
4. **A mature-audience skew** in content ratings
5. **US-centric production** with meaningful international diversification

These patterns are exactly what a content strategy team uses to plan
acquisitions and regional investment. This motivates **Part 2**: building an
actual model that classifies a title's audience group from its metadata.


---
# Part 2 — Algorithms Used for Prediction

Having understood the *what* and *why* through visualization, we now build a
model to *classify* each title's audience group. This section covers
feature preparation and the two algorithms used: **Logistic Regression**
and **Random Forest Classifier**.


## 4. Feature Encoding & Train-Test Split

We use `type`, `release_year`, `duration_value`, `num_genres`, and
`primary_country` (encoded) as inputs, with `audience_group` as the
3-class target (Kids/Family, Teens, Adults).


In [ ]:
model_df = df_clean.copy()

le_type = LabelEncoder()
model_df['type_enc'] = le_type.fit_transform(model_df['type'])

# Keep the top 15 countries explicit, group the rest as 'Other' to avoid
# an overly sparse one-hot encoding
top15 = model_df['primary_country'].value_counts().head(15).index
model_df['country_grouped'] = model_df['primary_country'].where(model_df['primary_country'].isin(top15), 'Other')
country_dummies = pd.get_dummies(model_df['country_grouped'], prefix='country', drop_first=True)

le_target = LabelEncoder()
model_df['audience_enc'] = le_target.fit_transform(model_df['audience_group'])

feature_df = pd.concat([
    model_df[['type_enc', 'release_year', 'duration_value', 'num_genres']],
    country_dummies
], axis=1)

X = feature_df
y = model_df['audience_enc']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Classes:", list(le_target.classes_))
print("Training set:", X_train.shape, " Test set:", X_test.shape)


## 5. Algorithm 1 — Logistic Regression

**Why:** Logistic Regression (in its multinomial form) is the standard
baseline for a multi-class classification problem like this one — fast,
interpretable, and a solid reference point for how much signal is in the
metadata alone.


In [ ]:
log_reg = LogisticRegression(max_iter=2000, random_state=42)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)

print("=== Logistic Regression Performance ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Precision (macro): {precision_score(y_test, y_pred_lr, average='macro'):.3f}")
print(f"Recall (macro)   : {recall_score(y_test, y_pred_lr, average='macro'):.3f}")
print(f"F1-score (macro) : {f1_score(y_test, y_pred_lr, average='macro'):.3f}")


## 6. Algorithm 2 — Random Forest Classifier

**Why:** Random Forest can capture non-linear interactions (e.g. certain
genres + duration ranges combining to signal "Adults") that a linear
decision boundary misses, and it ranks feature importance.


In [ ]:
rf_clf = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)

print("=== Random Forest Performance ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_rf):.3f}")
print(f"Precision (macro): {precision_score(y_test, y_pred_rf, average='macro'):.3f}")
print(f"Recall (macro)   : {recall_score(y_test, y_pred_rf, average='macro'):.3f}")
print(f"F1-score (macro) : {f1_score(y_test, y_pred_rf, average='macro'):.3f}")


## 7. Model Comparison — Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

labels = list(le_target.classes_)

sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Reds',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Logistic Regression — Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greys',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Random Forest — Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()


## 8. Feature Importance (Random Forest)

Which factors does the model rely on most to classify audience group?


In [ ]:
importances = pd.Series(rf_clf.feature_importances_, index=feature_df.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 6))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index, color='#E50914')
plt.title('Top 10 Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


**Conclusion:** `duration_value` and `release_year` tend to dominate as
top predictors, with `type` and specific countries contributing secondary
signal — content length and era are meaningfully tied to maturity rating.
This confirms the visual, exploratory analysis in Part 1 correctly framed
what shapes Netflix's catalog, and the classification model turns those
insights into a tool that could help auto-tag or sanity-check content
ratings for new acquisitions at scale.
